# Downstream Tasks and Model Verification


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from astropy.table import Table

#imports for plots

In [ ]:
from SpecML import load_specml
from Tokeniser import f, dq, w, valid_spectrum, valid_spectra, tokenize

In [ ]:
# ── Model + tokenisation setup — only this cell changes when you swap checkpoints
MODEL_FILE = 'SpecML 20260602 n20000 lr5e-4 4r 10PS 4OL.pt'

device = ('cuda' if torch.cuda.is_available() else
          'mps'  if torch.backends.mps.is_available() else 'cpu')

model, cfg = load_specml(MODEL_FILE, device=device)
print(cfg)   # shows patch_size, overlap, D_emb, n_layers etc. from the checkpoint

# Tokenise raw flux with the exact patch params the model was trained with
f_norm = (f - f.mean(axis=1, keepdims=True)) / f.std(axis=1, keepdims=True).clip(1e-10)
X, V, P = tokenize(f_norm, dq, w, cfg['patch_size'], cfg['overlap'], cfg['D_emb'])
print(f'X: {X.shape},  V: {V.shape},  P: {P.shape}')

### Loss Curve

In [ ]:
# Load the loss
loss = np.load('loss_curve.npy')

# Plot Loss
plt.title('Test 6: Loss Over 20000 Steps with a LR of 1e-4 over 4 Cosine Annealing Restarts')
plt.xlabel('Number of Steps')
plt.ylabel('Loss')
plt.yscale(f'log')
plt.plot(loss[::100], color = 'pink')
plt.savefig('Test 6 20260528 n20000 lr5e-4 4restarts')
plt.show()



### Spectrum Reconstruction

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
from Tokeniser import w


@torch.no_grad()
def reconstruct(model, cfg, flux_batch, valid_batch, masked=False, block_k=1):
    """
    Reconstruct spectra through SpecML, overlap-averaging patches back to pixel space.

    model      : loaded SpecML (from load_specml)
    cfg        : config dict returned by load_specml — supplies patch_size, overlap, D_emb
    flux_batch : (B, L) array/tensor — raw flux (post w^2 correction, pre-normalisation)
    valid_batch: (B, L) bool array/tensor — per-pixel DQ validity mask
    masked=False: single forward pass, all tokens visible
    masked=True : leave-one-block-out; costs N forward passes per batch
    """
    ps, ol = cfg['patch_size'], cfg['overlap']
    S      = ps - ol
    D_emb  = cfg['D_emb']

    if isinstance(flux_batch, torch.Tensor):
        flux_np  = flux_batch.cpu().numpy()
        valid_np = valid_batch.cpu().numpy().astype(bool)
    else:
        flux_np  = np.asarray(flux_batch, dtype=np.float64)
        valid_np = np.asarray(valid_batch, dtype=bool)

    B, L = flux_np.shape

    f_mean = np.nanmean(flux_np, axis=1, keepdims=True)
    f_std  = np.nanstd(flux_np,  axis=1, keepdims=True).clip(1e-10)
    f_norm = (flux_np - f_mean) / f_std

    x_t   = sliding_window_view(f_norm, ps, axis=1)[:, ::S]
    X_arr = np.concatenate([
        np.nanmean(x_t, axis=2, keepdims=True),
        np.nanstd( x_t, axis=2, keepdims=True),
        x_t,
    ], axis=2).astype(np.float32)

    dq_p  = sliding_window_view(valid_np, ps, axis=1)[:, ::S]
    V_arr = dq_p.all(axis=2)
    X_arr[~V_arr] = 0.0

    N      = X_arr.shape[1]
    L_used = (N - 1) * S + ps

    w_used    = w[:L_used]
    w_patches = sliding_window_view(w_used, ps)[::S].mean(axis=1)
    omegas    = 10000 ** (-2 * np.arange(D_emb // 2) / D_emb)
    product   = np.outer(w_patches * 1e4, omegas)
    P_enc     = np.empty((N, D_emb), dtype=np.float32)
    P_enc[:, 0::2] = np.sin(product)
    P_enc[:, 1::2] = np.cos(product)

    X_t = torch.from_numpy(X_arr).to(device)
    V_t = torch.from_numpy(V_arr).to(device)
    P_t = torch.from_numpy(P_enc).to(device)

    if not masked:
        out       = model.forward(X_t, V_t, P_t)
        recon_pat = out[:, :, 2:]
    else:
        recon_pat = torch.zeros(B, N, ps, device=device)
        half_lo, half_hi = (block_k - 1) // 2, block_k // 2
        for t in range(N):
            tlo, thi = max(0, t - half_lo), min(N - 1, t + half_hi)
            mp = X_t.clone()
            mp[:, tlo:thi+1, :] = 0.0
            out_t = model.forward(mp, V_t, P_t)
            recon_pat[:, t, :] = out_t[:, t, 2:]

    recon_pix = torch.zeros(B, L_used, device=device)
    count     = torch.zeros(B, L_used, device=device)
    for t in range(N):
        s = t * S
        recon_pix[:, s:s+ps] += recon_pat[:, t, :]
        count[:, s:s+ps]     += 1.0
    recon_pix /= count.clamp(min=1)

    mean_t = torch.from_numpy(f_mean[:, 0].astype(np.float32))
    std_t  = torch.from_numpy(f_std[:, 0].astype(np.float32))

    return dict(
        flux        = torch.from_numpy(flux_np[:, :L_used].astype(np.float32)),
        wave        = torch.from_numpy(w_used.astype(np.float32)),
        vmask       = torch.from_numpy(valid_np[:, :L_used]),
        flux_norm   = torch.from_numpy(f_norm[:, :L_used].astype(np.float32)),
        recon_norm  = recon_pix.cpu(),
        recon_flux  = (recon_pix.cpu() * std_t[:, None] + mean_t[:, None]),
        patches     = X_t.cpu(),
        recon_pat   = recon_pat.cpu(),
        token_vmask = V_t.cpu(),
        count       = count.cpu(),
        mean=mean_t, std=std_t,
        N=N, P=ps, S=S, L_used=L_used,
    )

In [ ]:
from Tokeniser import f, dq

# Pick 4 grade-3 spectra (catalog must be defined — run the Redshift section first,
# or replace with idx_show = np.arange(4) for a quick standalone check)
idx_show = np.where(mask_g3)[0][:4]
r = reconstruct(model, cfg, f[idx_show], dq[idx_show])

wave = r['wave'].numpy()

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
for i, ax in enumerate(axes):
    vmask = r['vmask'][i].numpy()
    ax.plot(wave, r['flux_norm'][i].numpy(),  color='steelblue', lw=0.8, label='input')
    ax.plot(wave, r['recon_norm'][i].numpy(), color='tomato',    lw=0.8, label='reconstruction')
    ax.fill_between(wave, -5, 5, where=~vmask, color='grey', alpha=0.2, label='invalid')
    ax.set_ylim(-5, 5)
    ax.set_ylabel('norm. flux')
    if i == 0:
        ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('wavelength (μm)')
fig.suptitle('Spectrum Reconstruction (normalised flux)')
plt.tight_layout()
plt.savefig('spectrum_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

### Redshift Prediction Plot

Load Data and Data Cleaning Process

In [ ]:
catalog = Table.read('dja_msaexp_emission_lines_v4.5.csv.gz', format='ascii')
catalog = catalog[catalog['grating'] == 'PRISM']
catalog = catalog[valid_spectrum][valid_spectra]   # align row-for-row with X / V

mask_g3 = np.array(
    (np.array(catalog['grade'].filled(0)) == 3) & (np.array(catalog['z_best']) >= 0)
)
z_g3 = torch.from_numpy(np.array(catalog['z_best'][mask_g3], dtype=np.float32))

Encode grade-3 spectra with frozen model

In [ ]:
with torch.no_grad():
    emb = model.encode(
        torch.from_numpy(X[mask_g3]).float().to(device),
        torch.from_numpy(V[mask_g3]).bool().to(device),
        torch.from_numpy(P).float().to(device),
    ).cpu()

# ---- 50/50 train/test split --------------------------------------------------
n = len(z_g3)
idx = torch.randperm(n, generator=torch.Generator().manual_seed(42))
split = n // 2

emb_train, emb_test = emb[idx[:split]], emb[idx[split:]]
z_train, z_test = z_g3[idx[:split]], z_g3[idx[split:]]

# ---- Normalise targets -------------------------------------------------------
z_mean, z_std = z_train.mean(), z_train.std()
z_train_n = (z_train - z_mean) / z_std

# ---- Linear head, encoder frozen --------------------------------------------
head = nn.Sequential(nn.LayerNorm(cfg['D_emb']), nn.Linear(cfg['D_emb'], 1))
opt = torch.optim.AdamW(head.parameters(), lr=1e-2)

head.train()
for step in range(2000):
    batch = torch.randint(len(emb_train), (256,))
    loss = F.mse_loss(head(emb_train[batch]).squeeze(-1), z_train_n[batch])
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 200 == 0:
        print(f'step {step:4d}  loss {loss.item():.4f}')

# ---- Evaluate ----------------------------------------------------------------
head.eval()
with torch.no_grad():
    z_pred = head(emb_test).squeeze(-1) * z_std + z_mean

dz = (z_pred - z_test).abs() / (1 + z_test)
print(f'grade-3 test set:  N={len(z_test)}')
print(f'MAE                {(z_pred - z_test).abs().mean().item():.4f}')
print(f'median |Δz|/(1+z)  {dz.median().item():.4f}')

Plotting Redshift Prediction

In [ ]:
z_true_np = z_test.numpy()
z_pred_np = z_pred.numpy()
dz_np = dz.numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
lim = (0, max(z_true_np.max(), z_pred_np.max()) * 1.05)
ax.scatter(
    z_true_np, z_pred_np, c=dz_np, cmap='plasma', s=8, alpha=0.7, vmin=0, vmax=0.1
)
ax.plot(lim, lim, 'k--', lw=0.8)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel('ha_EW_true')
ax.set_ylabel('ha_EW_pred')
ax.set_title('True vs predicted H_alpha Equivalent Widths')
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, 0.1))
fig.colorbar(sm, ax=ax, label='|Δz|/(1+z)')

ax = axes[1]
ax.hist(dz_np, bins=40, range=(0, 0.3), color='steelblue', edgecolor='none')
ax.axvline(
    float(dz.median()), color='red', lw=1.2, label=f'median={float(dz.median()):.4f}'
)
ax.set_xlabel('|Δz| / (1+z)')
ax.set_ylabel('count')
ax.set_title('Ha_EW error distribution')
ax.legend()

plt.tight_layout()
plt.savefig('downstream_linea_Ha_EW.png', dpi=150)
plt.show()
print('Saved downstream_linear_Ha_EW.png')

### Line Ha NII Prediction

In [ ]:
# ---- Ha+NII line flux prediction (uses catalog + X/V/P from setup cell) -----
snr_line = (np.array(catalog['line_ha_nii'],     dtype=np.float32) /
            np.array(catalog['line_ha_nii_err'],  dtype=np.float32).clip(1e-30))
mask_line = np.isfinite(snr_line) & (snr_line >= 3) & \
            (np.array(catalog['line_ha_nii'], dtype=np.float32) > 0)

line = torch.from_numpy(np.array(catalog['line_ha_nii'][mask_line], dtype=np.float32))

# ---- Encode with frozen model ------------------------------------------------
with torch.no_grad():
    emb = model.encode(
        torch.from_numpy(X[mask_line]).float().to(device),
        torch.from_numpy(V[mask_line]).bool().to(device),
        torch.from_numpy(P).float().to(device),
    ).cpu()

# ---- 50/50 train/test split --------------------------------------------------
n = len(line)
idx = torch.randperm(n, generator=torch.Generator().manual_seed(42))
split = n // 2

emb_train, emb_test = emb[idx[:split]], emb[idx[split:]]
line_train, line_test = line[idx[:split]], line[idx[split:]]

line_mean, line_std = line_train.mean(), line_train.std()
line_train_n = (line_train - line_mean) / line_std

# ---- Linear head, encoder frozen --------------------------------------------
head_line = nn.Sequential(nn.LayerNorm(cfg['D_emb']), nn.Linear(cfg['D_emb'], 1))
opt = torch.optim.AdamW(head_line.parameters(), lr=1e-2)

head_line.train()
for step in range(2000):
    batch = torch.randint(len(emb_train), (256,))
    loss = F.mse_loss(head_line(emb_train[batch]).squeeze(-1), line_train_n[batch])
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 200 == 0:
        print(f'step {step:4d}  loss {loss.item():.4f}')

# ---- Evaluate ----------------------------------------------------------------
head_line.eval()
with torch.no_grad():
    line_pred = head_line(emb_test).squeeze(-1) * line_std + line_mean

err = (line_pred - line_test).abs()
print(f'Ha+NII test set:  N={len(line_test)}')
print(f'MAE               {err.mean().item():.4f}')

# ---- Plot --------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
lt, lp = line_test.numpy(), line_pred.numpy()
lim = (min(lt.min(), lp.min()) * 1.1, max(lt.max(), lp.max()) * 1.1)
ax.scatter(lt, lp, c=err.numpy(), cmap='plasma', s=8, alpha=0.7)
ax.plot(lim, lim, 'k--', lw=0.8)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Ha+NII flux  (true)')
ax.set_ylabel('Ha+NII flux  (predicted)')
ax.set_title('True vs Predicted Ha+NII Line Flux')
fig.colorbar(plt.cm.ScalarMappable(cmap='plasma',
    norm=plt.Normalize(err.min().item(), err.max().item())), ax=ax, label='abs error')

ax = axes[1]
ax.hist(err.numpy(), bins=40, color='steelblue', edgecolor='none')
ax.axvline(float(err.median()), color='red', lw=1.2,
           label=f'median = {float(err.median()):.4f}')
ax.set_xlabel('|flux error|')
ax.set_ylabel('count')
ax.set_title('Ha+NII error distribution')
ax.legend()

plt.tight_layout()
plt.savefig('downstream_linear_ha_nii.png', dpi=150)
plt.show()
print('Saved downstream_linear_ha_nii.png')

In [18]:
import pandas as pd
import numpy as np

catalog = pd.read_csv('dja_msaexp_emission_lines_v4.5.csv.gz')
catalog_prism = catalog[catalog['grating'] == 'PRISM']


cols = catalog_prism.columns.tolist()
print([col for col in catalog_prism.columns if 'ha' in col])


# print(pd.isnull(catalog_prism['eqw_ha']).__len__)
# print(catalog_prism['eqw_ha'])


['line_ha', 'line_ha_err', 'eqw_ha_nii', 'line_ha_nii', 'line_ha_nii_err', 'eqw_ha']


/var/folders/zd/2rbx916x3vn4xr9snljfn9s9xmb1gn/T/ipykernel_4607/4011150888.py:4: DtypeWarning: Columns (0: valid, 1: ztype, 2: file_phot, 3: reviewer, 4: comment) have mixed types. Specify dtype option on import or set low_memory=False.
  catalog = pd.read_csv('dja_msaexp_emission_lines_v4.5.csv.gz')


### BPT Prediction

#### SFR, Gas Metallicity and Dust Attenuation (Balmer Decrament)

sfr - flux from something, multiply by something, using indicators  